# Conexión

In [4]:
import os
from dotenv import load_dotenv
from cassandra.cluster import Cluster
from cassandra.connection import DefaultEndPoint
from cassandra.auth import PlainTextAuthProvider

load_dotenv()

cassandra_user = os.getenv("CASSANDRA_USER")
cassandra_password = os.getenv("CASSANDRA_PASSWORD")
cassandra_endpoints = os.getenv("CASSANDRA_ENDPOINTS").split(',')

cassandra_nodes = []
for endpoint in cassandra_endpoints:
    host, port = endpoint.split(":")
    cassandra_nodes.append(DefaultEndPoint(host, int(port)))

print(f"🔗 Conectando a los nodos cargados por .env")

auth_provider = PlainTextAuthProvider(username=cassandra_user, password=cassandra_password)
cluster = Cluster(contact_points=cassandra_nodes, auth_provider=auth_provider)
session = cluster.connect()

print(f"✅ Conectado al clúster")

🔗 Conectando a los nodos cargados por .env
✅ Conectado al clúster


In [5]:
# 1. Crear Keyspace
keyspace_name = "jetstream_data"
session.execute(
    f"""
    CREATE KEYSPACE IF NOT EXISTS {keyspace_name} 
    WITH replication = {{
        'class': 'NetworkTopologyStrategy', 
        'dc1': 3
    }};
    """
)
session.set_keyspace(keyspace_name)
print(f"✅ Keyspace '{keyspace_name}' creado/seleccionado.")

# 2. Crear Tabla
# Partition Key: collection (agrupa los likes juntos)
# Clustering Key: time_us DESC (ordena los más recientes primero)
session.execute(
    """
    CREATE TABLE IF NOT EXISTS events (
        collection text,
        time_us bigint,
        did text,
        kind text,
        operation text,
        rkey text,
        record text,
        PRIMARY KEY ((collection), time_us, did)
    ) WITH CLUSTERING ORDER BY (time_us DESC);
    """
)
print("✅ Tabla 'events' creada.")

✅ Keyspace 'jetstream_data' creado/seleccionado.
✅ Tabla 'events' creada.


In [6]:
import asyncio
import websockets
import json
import logging
from cassandra.concurrent import execute_concurrent

# Configuración de logs para ver qué pasa
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# --- CONFIGURACIÓN ---
# Cambiamos a Follows y Blocks para un flujo de ~25 ops/s
COLLECTIONS = ["app.bsky.graph.follow", "app.bsky.graph.block"]
JETSTREAM_URL = f"wss://jetstream1.us-east.bsky.network/subscribe?{"&".join([f'wantedCollections={c}' for c in COLLECTIONS])}"

# Preparar el insert
insert_stmt = session.prepare("""
    INSERT INTO events (collection, time_us, did, kind, operation, rkey, record)
    VALUES (?, ?, ?, ?, ?, ?, ?)
""")

async def ingestor():
    reconnect_delay = 1
    while True:
        try:
            logging.info(f"🔗 Conectando a Jetstream...")
            async with websockets.connect(JETSTREAM_URL) as ws:
                logging.info("🚀 Ingesta iniciada. Presiona Ctrl+C o detén la celda para parar.")
                reconnect_delay = 1
                
                batch = []
                async for message in ws:
                    data = json.loads(message)
                    
                    # Estructuramos para Cassandra
                    commit = data.get("commit", {})
                    event_data = (
                        commit.get("collection"),
                        data.get("time_us"),
                        data.get("did"),
                        data.get("kind"),
                        commit.get("operation"),
                        commit.get("rkey"),
                        json.dumps(commit.get("record")) # Guardamos el record como JSON string
                    )
                    batch.append((insert_stmt, event_data))
                    
                    # Insertar en bloques de 20 para ser eficientes pero no pesados
                    if len(batch) >= 20:
                        execute_concurrent(session, batch, raise_on_first_error=False)
                        logging.info(f"📦 Batch de {len(batch)} eventos insertado.")
                        batch = []
                        
        except Exception as e:
            logging.error(f"❌ Error: {e}. Reintentando en {reconnect_delay}s...")
            await asyncio.sleep(reconnect_delay)
            reconnect_delay = min(reconnect_delay * 2, 60)

# Lanzar el ingestor
try:
    await ingestor()
except asyncio.CancelledError:
    logging.info("🛑 Ingesta detenida.")

2026-04-13 10:13:44,465 - 🔗 Conectando a Jetstream...
2026-04-13 10:13:45,077 - 🚀 Ingesta iniciada. Presiona Ctrl+C o detén la celda para parar.
2026-04-13 10:13:46,001 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:46,714 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:47,289 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:47,730 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:48,294 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:48,944 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:49,498 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:50,013 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:50,637 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:51,188 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:51,662 - 📦 Batch de 20 eventos insertado.
2026-04-13 10:13:51,733 - 🛑 Ingesta detenida.


In [7]:
import pandas as pd

# Consultar los últimos 10 registros
query = "SELECT collection, time_us, did, operation, record FROM jetstream_data.events LIMIT 10"
rows = session.execute(query)
df = pd.DataFrame(list(rows))

# Mostrar los datos bonitos
display(df)

,collection,time_us,did,operation,record
0,app.bsky.graph.follow,1776096831568930,did:plc:62kuirr4abdcujvgkg5zcz6v,delete,null
1,app.bsky.graph.follow,1776096831559607,did:plc:xvqbjvr7mvy3auir5ojjaawy,create,"{""$type"": ""app.bsky.graph.follow"", ""createdAt""..."
2,app.bsky.graph.follow,1776096831514988,did:plc:w2pyjru2yjf4mudhnfbpo74c,create,"{""$type"": ""app.bsky.graph.follow"", ""createdAt""..."
3,app.bsky.graph.follow,1776096831416299,did:plc:d2pi6trtljikqvoifxuytkar,delete,null
4,app.bsky.graph.follow,1776096831415426,did:plc:hmhdvkw7i5iddgpjebk5co5n,delete,null
5,app.bsky.graph.follow,1776096831394582,did:plc:ebn55hqjydwa2i7s23ht5mw2,create,"{""$type"": ""app.bsky.graph.follow"", ""createdAt""..."
6,app.bsky.graph.follow,1776096831326521,did:plc:3q2jfiphitk64wtdymsv2i7t,create,"{""$type"": ""app.bsky.graph.follow"", ""createdAt""..."
7,app.bsky.graph.follow,1776096831306677,did:plc:titsomriwhypsmmfftu3rxz6,create,"{""$type"": ""app.bsky.graph.follow"", ""createdAt""..."
8,app.bsky.graph.follow,1776096831215184,did:plc:lmmcastizqrlfcf62ryk5j46,delete,null
9,app.bsky.graph.follow,1776096831210715,did:plc:cxbe4h3guqgev7jsvov435wu,create,"{""$type"": ""app.bsky.graph.follow"", ""createdAt""..."
